# LightGBM

# 1. Imports & Setup

In [1]:
import pandas as pd
import numpy as np
np.random.seed(42)
from lightgbm import LGBMClassifier
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)

import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'lightgbm'

# 2. Load Dataset

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/train.csv")

# Check columns
df.columns

In [ ]:
df['sum_score'] = df.loc[:, 'A1_Score':'A10_Score'].sum(axis=1)

def convert_age(age):
    if age < 4: return 'Toddler'
    elif age < 12: return 'Kid'
    elif age < 18: return 'Teen'
    elif age < 40: return 'Young'
    else: return 'Senior'

df['ageGroup'] = df['age'].apply(convert_age)

df['ind'] = df['austim'] + df['used_app_before'] + df['jaundice']

# 3. Define Features (X) and Target (y)

In [ ]:
# Drop non-useful columns
X = df.drop(["Class/ASD", "result", "ID"], axis=1)
y = df["Class/ASD"]

print("X shape:", X.shape)
print("\nClass distribution:")
print(y.value_counts())

# 4. Encode Categorical Features

In [ ]:
categorical_cols = X.select_dtypes(include=["object"]).columns
print("Categorical columns:", categorical_cols)

X_encoded = X.copy()
from sklearn.preprocessing import LabelEncoder

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    encoders[col] = le

# 5. Train–Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

# 6. Feature Scaling (Required for Chi-Square)

In [ ]:
# Step 1: Handle Class Imbalance FIRST
from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)
X_train_res, y_train = ros.fit_resample(X_train, y_train)

# Step 2: Feature Scaling (CORRECT ORDER)
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train_res)   # Fit + Transform (TRAIN)
X_test_scaled = scaler.transform(X_test)             # Only Transform (TEST)

# 7. BASELINE MODEL — LightGBM (All Features)

In [ ]:
lgbm_base = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight=None,
    random_state=42,
    verbose=-1
)

lgbm_base.fit(X_train_scaled, y_train)

# Predictions
y_pred_base = lgbm_base.predict(X_test_scaled)
y_proba_base = lgbm_base.predict_proba(X_test_scaled)[:, 1]

print("Baseline Accuracy:", accuracy_score(y_test, y_pred_base))
print("Baseline ROC-AUC:", roc_auc_score(y_test, y_proba_base))

# 8. Feature Selection — SelectKBest (Chi-Square)

In [ ]:
selector = SelectKBest(score_func=chi2, k=10)

# Apply SelectKBest
X_train_kbest = selector.fit_transform(X_train_scaled, y_train)
X_test_kbest = selector.transform(X_test_scaled)

# Convert to DataFrame (FIX WARNING)
selected_features = X.columns[selector.get_support()]

X_train_kbest = pd.DataFrame(X_train_kbest, columns=selected_features)
X_test_kbest = pd.DataFrame(X_test_kbest, columns=selected_features)

# 9. LightGBM + SelectKBest Model

In [ ]:
# LightGBM + SelectKBest Model (FINAL FIXED VERSION)

lgbm_kbest = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight=None,   # FIXED: removed double balancing
    random_state=42,
    verbose=-1
)

# Train model
lgbm_kbest.fit(X_train_kbest, y_train)

# Cross Validation (Model Stability Check)
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    lgbm_kbest,
    X_train_kbest,
    y_train,
    cv=5
)

print("Cross Validation Mean Accuracy:", cv_scores.mean())

# Predictions
y_pred_kbest = lgbm_kbest.predict(X_test_kbest)
y_proba_kbest = lgbm_kbest.predict_proba(X_test_kbest)[:, 1]

# Evaluation Metrics
print("\nAccuracy (SelectKBest):", accuracy_score(y_test, y_pred_kbest))
print("ROC-AUC (SelectKBest):", roc_auc_score(y_test, y_proba_kbest))

print("\nClassification Report (SelectKBest):")
print(classification_report(y_test, y_pred_kbest))

# 10. Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_kbest,
    display_labels=["No ASD", "ASD"],
    cmap="Blues"
)

plt.title("Confusion Matrix – LightGBM + SelectKBest")
plt.show()

# 11. ROC Curve Comparison

In [ ]:
fpr_base, tpr_base, _ = roc_curve(y_test, y_proba_base)
fpr_kbest, tpr_kbest, _ = roc_curve(y_test, y_proba_kbest)

plt.figure(figsize=(7, 5))
plt.plot(fpr_base, tpr_base, label="LightGBM (All Features)")
plt.plot(fpr_kbest, tpr_kbest, label="LightGBM + SelectKBest")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

importances = lgbm_kbest.feature_importances_

plt.figure(figsize=(8,5))
plt.barh(selected_features, importances)
plt.xlabel("Importance Score")
plt.title("Feature Importance - LightGBM")
plt.show()

# 12. Final Result Comparison Table

In [ ]:
results = pd.DataFrame({
    "Model": [
        "LightGBM (All Features)",
        "LightGBM + SelectKBest"
    ],
    "Accuracy": [
        accuracy_score(y_test, y_pred_base),
        accuracy_score(y_test, y_pred_kbest)
    ],
    "ROC-AUC": [
        roc_auc_score(y_test, y_proba_base),
        roc_auc_score(y_test, y_proba_kbest)
    ]
})

results

In [ ]:
print("\n===== FINAL MODEL COMPARISON =====\n")

for i in range(len(results)):
    print(f"Model: {results['Model'][i]}")
    print(f"Accuracy: {results['Accuracy'][i]*100:.2f}%")
    print(f"ROC-AUC: {results['ROC-AUC'][i]:.4f}")
    print("-" * 40)

# Best model
best_index = results["ROC-AUC"].idxmax()

print("\n🏆 BEST MODEL:")
print(f"Model: {results['Model'][best_index]}")
print(f"Best Accuracy: {results['Accuracy'][best_index]*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

metrics = ['Precision', 'Recall', 'F1-Score', 'AUC']
scores = [0.934, 0.948, 0.941, 0.971]
accuracy = 94.38

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={'width_ratios': [1.5, 1]})
fig.patch.set_facecolor('#f8fbff') # Light background color

bars = ax1.barh(metrics, scores, color='#cd4646', height=0.5)
ax1.set_xlim(0.8, 1.0) # Start from 0.8 to highlight the differences
ax1.set_title('Performance Metrics', color='#cd4646', fontweight='bold', pad=15)
ax1.set_facecolor('#f8fbff')

# Removing spines (borders)
for spine in ['top', 'right']:
    ax1.spines[spine].set_visible(False)

# Adding value labels on bars
for bar in bars:
    width = bar.get_width()
    ax1.text(width + 0.002, bar.get_y() + bar.get_height()/2,
             f'{width:.3f}', va='center', fontweight='bold', color='#34495e')
ax2.set_facecolor('#f8fbff')
circle = plt.Circle((0.5, 0.5), 0.4, color='#cd4646', fill=False, linewidth=10)
inner_circle = plt.Circle((0.5, 0.5), 0.38, color='#e6d3d9', fill=True)

ax2.add_artist(inner_circle)
ax2.add_artist(circle)

# Adding Text inside the circle
ax2.text(0.5, 0.55, f'{accuracy}%', ha='center', va='center',
         fontsize=28, fontweight='bold', color='#cd4646')
ax2.text(0.5, 0.40, 'Accuracy', ha='center', va='center',
         fontsize=14, fontweight='bold', color='#34495e')

ax2.set_axis_off() # Hide axis for the circle

# --- BOTTOM TEXT BOX ---
footer_text = f"Accuracy: {accuracy}%  |  Precision: {scores[0]}  |  Recall: {scores[1]}  |  F1-Score: {scores[2]}  |  AUC: {scores[3]}"
plt.figtext(0.5, 0.02, footer_text, ha="center", fontsize=10,
            bbox={"facecolor":"white", "alpha":0.5, "pad":5, "edgecolor":"#cd4646", "boxstyle":"round,pad=0.5"},
            color='#34495e')

plt.tight_layout(rect=[0, 0.05, 1, 0.95])
plt.show()

In [ ]:
# ==============================
# QUESTION SET (AQ-10)
# ==============================

questions = {
    "A1_Score": "Does the child look at you when you call his/her name?",
    "A2_Score": "How easy is it for the child to make eye contact?",
    "A3_Score": "Does the child point to indicate interest?",
    "A4_Score": "Does the child pretend play?",
    "A5_Score": "Does the child follow where you're looking?",
    "A6_Score": "Does the child show empathy?",
    "A7_Score": "Does the child use gestures like waving?",
    "A8_Score": "Does the child stare at nothing?",
    "A9_Score": "Does the child have speech difficulty?",
    "A10_Score": "Does the child show repetitive behavior?"
}

# Reverse scoring (important)
reverse_questions = ["A1_Score","A2_Score","A3_Score","A4_Score","A5_Score","A6_Score","A7_Score"]

# ==============================
# AGE CONVERSION FUNCTION
# ==============================

def convert_age(age):
    if age < 4: return 'Toddler'
    elif age < 12: return 'Kid'
    elif age < 18: return 'Teen'
    elif age < 40: return 'Young'
    else: return 'Senior'


# ==============================
# USER PREDICTION FUNCTION
# ==============================

def predict_autism():
    print("\n===== AUTISM SCREENING TEST =====\n")

    user_data = {}

    # Questionnaire input
    for key, question in questions.items():
        while True:
            ans = input(f"{question} (yes/no): ").lower()

            if ans in ["yes", "no"]:
                if key in reverse_questions:
                    user_data[key] = 1 if ans == "no" else 0
                else:
                    user_data[key] = 1 if ans == "yes" else 0
                break
            else:
                print("Enter yes or no only")

    # Basic details
    age = int(input("Age: "))
    user_data["age"] = age
    user_data["ageGroup"] = convert_age(age)

    # IMPORTANT FIX
    user_data["age_desc"] = "12-16 years"

    user_data["gender"] = input("Gender (m/f): ")
    user_data["ethnicity"] = input("Ethnicity: ")
    user_data["jaundice"] = int(input("Jaundice (0/1): "))
    user_data["austim"] = int(input("Family history (0/1): "))
    user_data["used_app_before"] = int(input("Used app before (0/1): "))

    # FIXED COLUMN NAME
    user_data["contry_of_res"] = input("Country: ")

    user_data["relation"] = input("Relation: ")

    # Convert to DataFrame
    user_df = pd.DataFrame([user_data])

    # Feature Engineering
    user_df['sum_score'] = user_df.loc[:, 'A1_Score':'A10_Score'].sum(axis=1)
    user_df['ind'] = user_df['austim'] + user_df['used_app_before'] + user_df['jaundice']

    # Encode categorical
    for col in categorical_cols:
        if col in user_df.columns:
            try:
                user_df[col] = encoders[col].transform(user_df[col].astype(str))
            except:
                user_df[col] = 0

    # 🔥 AUTO FIX: Missing columns handle
    for col in X.columns:
        if col not in user_df.columns:
            user_df[col] = 0

    # Ensure same column order
    user_df = user_df[X.columns]

    # Scale
    user_scaled = scaler.transform(user_df)

    # Feature selection
    user_kbest = selector.transform(user_scaled)

    # Prediction
    prediction = lgbm_kbest.predict(user_kbest)[0]
    probability = lgbm_kbest.predict_proba(user_kbest)[0][1]

    print("\n===== RESULT =====")

    if prediction == 1:
        print(f"⚠️ Autism Likely (Probability: {probability:.2f})")
    else:
        print(f"✅ No Autism (Probability: {probability:.2f})")


# ==============================
# RUN
# ==============================

predict_autism()


===== AUTISM SCREENING TEST =====

